In [3]:
import sys
from pathlib import Path

from xgboost import XGBClassifier
from openbb import obb

repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "test_001_nvda").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from backtesting.test_001_nvda.config import build_model_path, load_strategy_config

strategy_config = load_strategy_config(
    repo_root / "backtesting" / "test_001_nvda" / "strategy_config.json"
 )
df = obb.equity.price.historical(
    strategy_config["stock_symbol"], provider=strategy_config["data_src"]
).to_df()

df["return"] = df["close"].pct_change()
df["vol_change"] = df["volume"].pct_change()
df["target"] = (df["return"].shift(-1) > 0).astype(int)

features = ["return", "vol_change"]
X = df[features].dropna()[:-1]
y = df["target"].loc[X.index]

model_path = build_model_path(repo_root, strategy_config["model_name"])
model = XGBClassifier(
    n_estimators=10,
    max_depth=3,
    learning_rate=0.1,
 )
model.fit(X, y)

model.save_model(model_path)
print(f"{strategy_config['model_name']}.json generated in {model_path.parent}/")

test-001-nvda.json generated in /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/


In [4]:
import numpy as np
import pandas as pd
from openbb import obb

rl_df = obb.equity.price.historical(
    strategy_config["stock_symbol"], provider=strategy_config["data_src"]
).to_df().copy()

macd_fast = rl_df["close"].ewm(span=12, adjust=False).mean()
macd_slow = rl_df["close"].ewm(span=26, adjust=False).mean()
rl_df["macd_line"] = macd_fast - macd_slow
rl_df["macd_signal"] = rl_df["macd_line"].ewm(span=9, adjust=False).mean()
rl_df["macd_hist"] = rl_df["macd_line"] - rl_df["macd_signal"]

delta = rl_df["close"].diff()
gains = delta.clip(lower=0)
losses = -delta.clip(upper=0)
avg_gain = gains.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
avg_loss = losses.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
relative_strength = avg_gain / avg_loss.replace(0, np.nan)
rl_df["rsi"] = 100 - (100 / (1 + relative_strength))

typical_price = (rl_df["high"] + rl_df["low"] + rl_df["close"]) / 3
cci_mean = typical_price.rolling(20).mean()
cci_mad = typical_price.rolling(20).apply(
    lambda values: np.mean(np.abs(values - values.mean())), raw=True
)
rl_df["cci"] = (typical_price - cci_mean) / (0.015 * cci_mad)

rl_df["buy_signal"] = (
    (rl_df["macd_line"] > rl_df["macd_signal"])
) & (
    rl_df["macd_line"].shift(1) <= rl_df["macd_signal"].shift(1)
) & (rl_df["rsi"] > 50)

rl_df["sell_signal"] = (
    (rl_df["macd_line"] < rl_df["macd_signal"])
) & (
    rl_df["macd_line"].shift(1) >= rl_df["macd_signal"].shift(1)
) & (rl_df["rsi"] < 50)

rl_df = rl_df.dropna().copy()
rl_df["next_return"] = rl_df["close"].pct_change().shift(-1)
rl_df = rl_df.iloc[:-1].copy()

state_columns = [
    "macd_line",
    "macd_signal",
    "macd_hist",
    "rsi",
    "cci",
    "position_flag",
]
action_names = ["hold", "buy", "sell"]
fee_bps = 5
transaction_cost = fee_bps / 10_000
discount_gamma = 0.95

records = []
for row_number in range(len(rl_df) - 1):
    current_row = rl_df.iloc[row_number]
    next_row = rl_df.iloc[row_number + 1]
    next_return = current_row["next_return"]

    for position_flag in (0, 1):
        valid_actions = ["hold"]
        if position_flag == 0 and bool(current_row["buy_signal"]):
            valid_actions.append("buy")
        if position_flag == 1 and bool(current_row["sell_signal"]):
            valid_actions.append("sell")

        for action_name in valid_actions:
            next_position_flag = position_flag
            reward = 0.0

            if action_name == "buy":
                next_position_flag = 1
                reward = next_return - transaction_cost
            elif action_name == "hold":
                reward = next_return if position_flag == 1 else 0.0
            elif action_name == "sell":
                next_position_flag = 0
                reward = -transaction_cost

            record = {
                "state_row": row_number,
                "date": str(rl_df.index[row_number]),
                "action_name": action_name,
                "reward": reward,
                "buy_signal": int(bool(current_row["buy_signal"])),
                "sell_signal": int(bool(current_row["sell_signal"])),
                "next_buy_signal": int(bool(next_row["buy_signal"])),
                "next_sell_signal": int(bool(next_row["sell_signal"])),
                "next_position_flag": next_position_flag,
            }

            for column_name in state_columns:
                if column_name == "position_flag":
                    record[column_name] = position_flag
                    record[f"next_{column_name}"] = next_position_flag
                else:
                    record[column_name] = current_row[column_name]
                    record[f"next_{column_name}"] = next_row[column_name]

            records.append(record)

transitions = pd.DataFrame(records)
transitions[["date", "action_name", "reward"]].head(), transitions["action_name"].value_counts()

(         date action_name    reward
 0  2025-04-16        hold  0.000000
 1  2025-04-16        hold -0.028711
 2  2025-04-17        hold  0.000000
 3  2025-04-17        hold -0.045128
 4  2025-04-21        hold  0.000000,
 action_name
 hold    462
 buy       8
 sell      7
 Name: count, dtype: int64)

In [5]:
from xgboost import XGBRegressor

def score_valid_actions(feature_frame, buy_signal, sell_signal, position_flag, models):
    zero_scores = np.zeros(len(feature_frame))
    scores = pd.DataFrame(index=feature_frame.index)

    hold_model = models.get("hold")
    buy_model = models.get("buy")
    sell_model = models.get("sell")

    scores["hold"] = zero_scores if hold_model is None else hold_model.predict(feature_frame)

    buy_scores = zero_scores if buy_model is None else buy_model.predict(feature_frame)
    sell_scores = zero_scores if sell_model is None else sell_model.predict(feature_frame)

    scores["buy"] = np.where(
        (position_flag == 0) & (buy_signal == 1),
        buy_scores,
        -np.inf,
    )
    scores["sell"] = np.where(
        (position_flag == 1) & (sell_signal == 1),
        sell_scores,
        -np.inf,
    )
    return scores

action_models = {name: None for name in action_names}
training_targets = transitions["reward"].copy()

for iteration in range(8):
    next_feature_frame = transitions[[f"next_{column}" for column in state_columns]].copy()
    next_feature_frame.columns = state_columns

    next_scores = score_valid_actions(
        next_feature_frame,
        transitions["next_buy_signal"],
        transitions["next_sell_signal"],
        transitions["next_position_flag"],
        action_models,
    )
    max_next_q = next_scores.max(axis=1).replace(-np.inf, 0.0)
    training_targets = transitions["reward"] + discount_gamma * max_next_q

    for action_name in action_names:
        action_slice = transitions[transitions["action_name"] == action_name]
        if action_slice.empty:
            continue

        model = XGBRegressor(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
        )
        model.fit(action_slice[state_columns], training_targets.loc[action_slice.index])
        action_models[action_name] = model

state_catalog = transitions[[
    "state_row",
    "date",
    *state_columns,
    "buy_signal",
    "sell_signal",
]].drop_duplicates().reset_index(drop=True)
policy_scores = score_valid_actions(
    state_catalog[state_columns],
    state_catalog["buy_signal"],
    state_catalog["sell_signal"],
    state_catalog["position_flag"],
    action_models,
)
state_catalog["chosen_action"] = policy_scores.idxmax(axis=1)

importance_tables = {}
for action_name, model in action_models.items():
    if model is None:
        continue
    importance_tables[action_name] = pd.Series(
        model.feature_importances_,
        index=state_columns,
        name=action_name,
    ).sort_values(ascending=False)

state_catalog[["date", "position_flag", "buy_signal", "sell_signal", "chosen_action"]].head(), importance_tables

(         date  position_flag  buy_signal  sell_signal chosen_action
 0  2025-04-16              0           0            0          hold
 1  2025-04-16              1           0            0          hold
 2  2025-04-17              0           0            0          hold
 3  2025-04-17              1           0            0          hold
 4  2025-04-21              0           0            1          hold,
 {'hold': position_flag    0.252556
  rsi              0.247752
  macd_signal      0.157922
  macd_line        0.151510
  macd_hist        0.124188
  cci              0.066072
  Name: hold, dtype: float32,
  'buy': cci              0.431276
  macd_hist        0.195801
  macd_line        0.149348
  rsi              0.135897
  macd_signal      0.087679
  position_flag    0.000000
  Name: buy, dtype: float32,
  'sell': cci              0.355632
  rsi              0.322594
  macd_hist        0.190725
  macd_signal      0.067252
  macd_line        0.063797
  position_flag    0.000000

In [6]:
import json

rl_model_paths = {}
for action_name, model in action_models.items():
    if model is None:
        continue
    artifact_path = model_path.parent / f"{strategy_config['model_name']}-q-{action_name}.json"
    model.save_model(artifact_path)
    rl_model_paths[action_name] = str(artifact_path)

metadata_path = model_path.parent / f"{strategy_config['model_name']}-policy-metadata.json"
policy_metadata = {
    "model_name": strategy_config["model_name"],
    "policy_type": "fitted_q_iteration",
    "actions": {"hold": 0, "buy": 1, "sell": 2},
    "state_columns": state_columns,
    "gamma": discount_gamma,
    "transaction_cost_bps": fee_bps,
    "indicator_parameters": {
        "macd_fast": 12,
        "macd_slow": 26,
        "macd_signal": 9,
        "rsi_period": 14,
        "cci_period": 20,
    },
    "gate_rules": {
        "buy": "macd_line crosses above macd_signal and rsi > 50 while flat",
        "sell": "macd_line crosses below macd_signal and rsi < 50 while long",
        "hold": "always valid",
    },
    "artifacts": rl_model_paths,
}

with metadata_path.open("w") as metadata_file:
    json.dump(policy_metadata, metadata_file, indent=2)

print("Saved RL artifacts:")
for action_name, artifact_path in rl_model_paths.items():
    print(f"  {action_name}: {artifact_path}")
print(f"  metadata: {metadata_path}")

Saved RL artifacts:
  hold: /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/test-001-nvda-q-hold.json
  buy: /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/test-001-nvda-q-buy.json
  sell: /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/test-001-nvda-q-sell.json
  metadata: /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/test-001-nvda-policy-metadata.json
